# 164. Multi-Token Prediction：怎样实现多步预测头、边界 mask 与推理解码？

> **面试问题：MTP 为什么可能提高训练信号和推理速度？多 horizon 标签、loss、候选树和 verifier 怎样实现？**

## 先给结论

MTP 在同一位置预测未来多个 token，但不能把目标简单右移后忽略文档边界。训练要明确每个 horizon 的标签与有效 mask、主 next-token 目标的权重、共享干路的梯度；推理加速则还需要候选组织和目标模型验证，训练准确率高不自动等于无偏加速。

## 推荐的回答主线

1. 先定义第 h 个头在位置 t 预测 `x[t+h]`，处理 padding、EOS 与 packed-document 边界。
2. 显式实现共享 causal 表示、独立预测头和按有效 token 归一化的加权交叉熵。
3. 推理时把多头结果看成草稿候选，目标分布仍需验证；区分候选命中与最终采样正确性。
4. 用 per-horizon accuracy、accepted tokens、额外 FLOPs、端到端延迟和质量门禁共同评估。

## 本 Notebook 的实现边界

教学模型用前缀均值作为简单 causal backbone，目的是隔离 MTP 的目标、mask、梯度和候选语义；它不是 DeepSeek-V3 的完整 MTP 模块，也没有实现真实 speculative kernel。

## 一手资料

- [Better & Faster LLMs via Multi-token Prediction](https://arxiv.org/abs/2404.19737)
- [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437)
- [Medusa](https://arxiv.org/abs/2401.10774)


In [ ]:
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
from dataclasses import dataclass, asdict  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。

# 使用两段 packed 文档构造边界反例，0 只在 padding 中出现。
torch.manual_seed(164)  # 执行当前语句以推进本节示例。
VOCAB, D_MODEL, HORIZONS = 17, 12, 3  # 计算并保存当前步骤的中间状态。
tokens = torch.tensor([[1, 2, 3, 4, 5, 6, 0], [7, 8, 9, 10, 11, 0, 0]])  # 计算并保存当前步骤的中间状态。
valid = tokens.ne(0)  # 计算并保存当前步骤的中间状态。
doc_ids = torch.tensor([[10, 10, 10, 10, 20, 20, -1], [30, 30, 30, 30, 40, -1, -1]])  # 计算并保存当前步骤的中间状态。

assert tokens.shape == valid.shape == doc_ids.shape  # 用受控断言验证关键不变量。
assert HORIZONS >= 2  # 用受控断言验证关键不变量。
assert valid.sum().item() == 11  # 用受控断言验证关键不变量。


## 1. 生成多 horizon 标签：不能跨 packed-document 偷看

令 horizon=1 表示标准 next-token，horizon=2/3 表示更远未来。标签位置超界、任一端是 padding、或源/目标属于不同文档时都必须 mask；否则模型会学习一段文档结尾到下一段开头的伪转移。


In [ ]:
def make_mtp_targets(token_ids, valid_mask, document_ids, horizons):  # 定义本节可复用的核心函数。
    b, length = token_ids.shape  # 计算并保存当前步骤的中间状态。
    targets = torch.zeros(b, length, horizons, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    mask = torch.zeros(b, length, horizons, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    for h in range(1, horizons + 1):  # 遍历输入元素以累积或检查结果。
        targets[:, :-h, h - 1] = token_ids[:, h:]  # 计算并保存当前步骤的中间状态。
        same_document = document_ids[:, :-h].eq(document_ids[:, h:])  # 计算并保存当前步骤的中间状态。
        ok = valid_mask[:, :-h] & valid_mask[:, h:] & same_document  # 计算并保存当前步骤的中间状态。
        mask[:, :-h, h - 1] = ok  # 计算并保存当前步骤的中间状态。
    return targets, mask  # 返回当前分支计算出的结果。

# 位置 3 的下一 token 属于新文档，必须无监督；位置 0 的三个 horizon 都在同一文档。
targets, mtp_mask = make_mtp_targets(tokens, valid, doc_ids, HORIZONS)  # 计算并保存当前步骤的中间状态。
assert targets.shape == (2, 7, HORIZONS)  # 用受控断言验证关键不变量。
assert mtp_mask[0, 0].tolist() == [True, True, True]  # 用受控断言验证关键不变量。
assert not mtp_mask[0, 3, 0]  # 用受控断言验证关键不变量。


## 2. 手写共享 causal backbone 与独立预测头

真实模型会复用 Transformer 隐状态并增加轻量预测模块。这里用 token embedding 的前缀均值保证位置 t 只依赖 `<=t`，随后每个 horizon 使用独立线性头。这样可以直接检查形状和因果性，而不借助高层 Trainer。


In [ ]:
class TinyMTP(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab, dim, horizons):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)  # 计算并保存当前步骤的中间状态。
        self.trunk = nn.Sequential(nn.Linear(dim, dim), nn.Tanh())  # 计算并保存当前步骤的中间状态。
        self.heads = nn.ModuleList(nn.Linear(dim, vocab, bias=False) for _ in range(horizons))  # 计算并保存当前步骤的中间状态。

    def forward(self, token_ids, valid_mask):  # 定义本节可复用的核心函数。
        emb = self.embedding(token_ids)  # 计算并保存当前步骤的中间状态。
        prefix_sum = torch.cumsum(emb * valid_mask[..., None], dim=1)  # 计算并保存当前步骤的中间状态。
        prefix_count = torch.cumsum(valid_mask, dim=1).clamp_min(1)[..., None]  # 计算并保存当前步骤的中间状态。
        hidden = self.trunk(prefix_sum / prefix_count)  # 计算并保存当前步骤的中间状态。
        return torch.stack([head(hidden) for head in self.heads], dim=2)  # 返回当前分支计算出的结果。

# 修改未来 token 不应改变更早位置的 causal logits。
model = TinyMTP(VOCAB, D_MODEL, HORIZONS)  # 计算并保存当前步骤的中间状态。
logits = model(tokens, valid)  # 计算并保存当前步骤的中间状态。
changed = tokens.clone(); changed[0, 5] = 12  # 计算并保存当前步骤的中间状态。
changed_logits = model(changed, valid)  # 计算并保存当前步骤的中间状态。
assert logits.shape == (2, 7, HORIZONS, VOCAB)  # 用受控断言验证关键不变量。
assert torch.allclose(logits[0, :5], changed_logits[0, :5])  # 用受控断言验证关键不变量。
assert len(model.heads) == HORIZONS  # 用受控断言验证关键不变量。


## 3. 按有效 token 归一化 loss，保留主目标锚点

不同 horizon 的有效 token 数不同，不能先把 padding 当零损失再对固定矩阵求平均。下面分别按 mask 归一化，再用显式权重合并；通常 horizon=1 是主语言建模目标，更远目标作为辅助项。


In [ ]:
def mtp_cross_entropy(all_logits, all_targets, all_mask, weights):  # 定义本节可复用的核心函数。
    per_head, total = [], all_logits.new_tensor(0.0)  # 计算并保存当前步骤的中间状态。
    for h, weight in enumerate(weights):  # 遍历输入元素以累积或检查结果。
        flat_loss = F.cross_entropy(all_logits[:, :, h].reshape(-1, VOCAB), all_targets[:, :, h].reshape(-1), reduction="none")  # 计算并保存当前步骤的中间状态。
        head_mask = all_mask[:, :, h].reshape(-1)  # 计算并保存当前步骤的中间状态。
        head_loss = flat_loss[head_mask].mean()  # 计算并保存当前步骤的中间状态。
        per_head.append(head_loss)  # 执行当前语句以推进本节示例。
        total = total + weight * head_loss  # 计算并保存当前步骤的中间状态。
    return total, torch.stack(per_head)  # 返回当前分支计算出的结果。

# 总损失应精确等于各头损失加权和，且每个头都有有效监督。
weights = torch.tensor([1.0, 0.4, 0.2])  # 计算并保存当前步骤的中间状态。
loss, head_losses = mtp_cross_entropy(logits, targets, mtp_mask, weights)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(loss, (head_losses * weights).sum())  # 用受控断言验证关键不变量。
assert torch.isfinite(head_losses).all()  # 用受控断言验证关键不变量。
assert all(mtp_mask[:, :, h].any() for h in range(HORIZONS))  # 用受控断言验证关键不变量。


## 4. 梯度合同：辅助头应更新共享干路，但不能覆盖主目标

MTP 的价值之一是让每个位置收到更多未来监督。验证时不仅看 loss，还要确认辅助目标确实给共享 trunk 梯度、各头参数彼此独立，以及关闭辅助权重后主头梯度仍存在。


In [ ]:
# 先清梯度再反传，检查共享与各预测头都收到有限梯度。
model.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
loss.backward()  # 执行当前语句以推进本节示例。
trunk_grad = model.trunk[0].weight.grad  # 计算并保存当前步骤的中间状态。
head_grad_norms = [head.weight.grad.norm().item() for head in model.heads]  # 计算并保存当前步骤的中间状态。
assert trunk_grad is not None and torch.isfinite(trunk_grad).all()  # 用受控断言验证关键不变量。
assert all(value > 0 for value in head_grad_norms)  # 用受控断言验证关键不变量。
assert model.heads[0].weight.data_ptr() != model.heads[1].weight.data_ptr()  # 用受控断言验证关键不变量。


## 5. 候选树：多头输出是提案，不是已经被目标模型接受的 token

独立 horizon top-k 的笛卡尔积可形成浅层候选树，但会快速膨胀。实际系统会限制分支、复用前缀，并由目标模型一次验证多个节点。下面仅构造有界候选并保留对数概率分数。


In [ ]:
def candidate_paths(position_logits, branch=2, max_paths=8):  # 定义本节可复用的核心函数。
    per_horizon = []  # 计算并保存当前步骤的中间状态。
    for h in range(position_logits.shape[0]):  # 遍历输入元素以累积或检查结果。
        logp = position_logits[h].log_softmax(-1)  # 计算并保存当前步骤的中间状态。
        values, ids = torch.topk(logp, branch)  # 计算并保存当前步骤的中间状态。
        per_horizon.append(list(zip(ids.tolist(), values.tolist())))  # 执行当前语句以推进本节示例。
    paths = [([], 0.0)]  # 计算并保存当前步骤的中间状态。
    for choices in per_horizon:  # 遍历输入元素以累积或检查结果。
        paths = [(prefix + [token], score + value) for prefix, score in paths for token, value in choices]  # 计算并保存当前步骤的中间状态。
        paths = sorted(paths, key=lambda item: item[1], reverse=True)[:max_paths]  # 计算并保存当前步骤的中间状态。
    return paths  # 返回当前分支计算出的结果。

# 分支数必须受预算约束，路径长度等于 horizon 数，分数按降序排列。
paths = candidate_paths(logits[0, 1], branch=2, max_paths=5)  # 计算并保存当前步骤的中间状态。
assert len(paths) == 5  # 用受控断言验证关键不变量。
assert all(len(path) == HORIZONS for path, _ in paths)  # 用受控断言验证关键不变量。
assert all(paths[i][1] >= paths[i + 1][1] for i in range(len(paths) - 1))  # 用受控断言验证关键不变量。


## 6. Verifier 接受前缀：首个错误后不能继续提交

用贪心 oracle 演示验证协议：候选路径从左到右与目标模型决定比较，只提交最长一致前缀。真实随机采样还需要接受—拒绝或残差分布校正；仅比较 argmax 不能证明生成分布无偏。


In [ ]:
def accepted_prefix(proposal, target_tokens):  # 定义本节可复用的核心函数。
    accepted = []  # 计算并保存当前步骤的中间状态。
    for proposed, target in zip(proposal, target_tokens):  # 遍历输入元素以累积或检查结果。
        if proposed != target:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        accepted.append(proposed)  # 执行当前语句以推进本节示例。
    return accepted  # 返回当前分支计算出的结果。

# 即便错误后的 token 碰巧正确，也不能越过首个分歧继续接受。
target_path = [4, 8, 2]  # 计算并保存当前步骤的中间状态。
assert accepted_prefix([4, 8, 2], target_path) == target_path  # 用受控断言验证关键不变量。
assert accepted_prefix([4, 7, 2], target_path) == [4]  # 用受控断言验证关键不变量。
assert accepted_prefix([9, 8, 2], target_path) == []  # 用受控断言验证关键不变量。


## 7. 评测：同时报告各 horizon 准确率与接受长度

远期预测更难，不能只报平均 token accuracy。离线至少按 horizon、文档类型和长度切片；线上要报每次 target forward 接受 token 数、拒绝率、额外 head FLOPs、端到端吞吐和 p99 延迟。


In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def horizon_accuracy(all_logits, all_targets, all_mask):  # 定义本节可复用的核心函数。
    pred = all_logits.argmax(-1)  # 计算并保存当前步骤的中间状态。
    result = []  # 计算并保存当前步骤的中间状态。
    for h in range(all_logits.shape[2]):  # 遍历输入元素以累积或检查结果。
        mask_h = all_mask[:, :, h]  # 计算并保存当前步骤的中间状态。
        result.append((pred[:, :, h][mask_h] == all_targets[:, :, h][mask_h]).float().mean().item())  # 计算并保存当前步骤的中间状态。
    return result  # 返回当前分支计算出的结果。

# 指标必须落在概率范围内，且样本权重使用真实有效 token 数。
accuracies = horizon_accuracy(logits.detach(), targets, mtp_mask)  # 计算并保存当前步骤的中间状态。
valid_counts = mtp_mask.sum((0, 1)).tolist()  # 计算并保存当前步骤的中间状态。
assert len(accuracies) == HORIZONS  # 用受控断言验证关键不变量。
assert all(0.0 <= value <= 1.0 for value in accuracies)  # 用受控断言验证关键不变量。
assert valid_counts[0] >= valid_counts[1] >= valid_counts[2]  # 用受控断言验证关键不变量。


## 8. 版本与发布门禁：训练头、解码器和 tokenizer 是同一合同

若 horizon 数、tokenizer 或候选树策略不一致，服务仍可能运行但标签语义已经错位。把它们写进制品摘要，并用固定 prompt 做主头质量、候选覆盖、accepted-token 和延迟的成对回归。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class MTPArtifact:  # 定义承载本节状态与行为的数据结构。
    vocab_size: int  # 执行当前语句以推进本节示例。
    horizons: int  # 执行当前语句以推进本节示例。
    tokenizer_hash: str  # 执行当前语句以推进本节示例。
    decoder_policy: str  # 执行当前语句以推进本节示例。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# 任一接口字段变化都必须产生新摘要，加载时拒绝旧 decoder 配置。
artifact = MTPArtifact(VOCAB, HORIZONS, "tok-demo-v1", "tree-top2-v1")  # 计算并保存当前步骤的中间状态。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert len(digest) == 64  # 用受控断言验证关键不变量。
assert digest == artifact_hash(artifact)  # 用受控断言验证关键不变量。
assert digest != artifact_hash(MTPArtifact(VOCAB, HORIZONS + 1, "tok-demo-v1", "tree-top2-v1"))  # 用受控断言验证关键不变量。


## 面试收束：怎样把实现讲成工程答案

建议按“目标与约束 → 数据/张量合同 → 核心公式 → 正确性 oracle → 性能与安全边界 → 发布门禁”作答。Notebook 里的小张量和受控状态机只证明机制成立，不等于真实集群吞吐、真实模型质量或生产安全性。上线前还要补齐目标硬件 profiling、故障注入、分布式一致性、真实数据切片、权限审计、版本化制品和回滚演练。

可继续追问：规模扩大后哪个状态最贵？哪条等价性可作为回归测试？输入或版本变化时怎样拒绝静默错误？指标改善是否只是成本、数据污染或评测器偏差造成的？
